# 13 — Bài báo, bước 1: phân tích lại dự đoán đã lưu của GĐ10 (máy local)

**Phạm vi**

Các phân tích bổ sung cho bài báo về phân loại, pre-register ở amendment v3.1 trước khi tính: kiểm tra trùng lặp nội dung
giữa các tập, chấm lại trên tập khử trùng, baseline phẳng trên Lazada, kiểm định McNemar, phân tích lỗi theo lớp và biên
Pareto giữa hiệu năng và chi phí.

**Ràng buộc**

Chỉ dùng dự đoán đã lưu `gd10_*` trong `05_experiments/rq1_classification/`. Không train lại, không gọi API, không đổi
config, prompt hay ngưỡng, không chọn lại cấu hình. Nhãn gold của Hold-out, focused và Lazada được đọc lại chỉ để chấm
lại các dự đoán đã khóa, ghi trong `01_data/HOLDOUT_LOCK.md`.

**Điều kiện dừng**

Trước mọi phân tích, notebook phải tái tạo đúng các bảng GĐ10 trên tập đầy đủ. Lệch bất kỳ ô nào thì dừng.

In [1]:
import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.final_eval import load_frozen
from src.classification.evaluate import FOUR_CLASSES
from src.classification import paper_analysis as PA
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
FZ = load_frozen(ROOT)
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT)
SETS = {'Train': 'shopee_train.csv', 'Validation': 'shopee_validation.csv', 'Hold-out': 'shopee_holdout.csv',
        'Focused': 'shopee_focused_feature_ux.csv', 'Lazada': 'lazada_external_test.csv'}
D = {k: pd.read_csv(os.path.join(DATA, f), usecols=['review_id', 'text', 'gold_label'], encoding='utf-8-sig') for k, f in SETS.items()}
D['Train ∪ Validation'] = pd.concat([D['Train'], D['Validation']], ignore_index=True)
for k, d in D.items():
    print(f'{k}: {len(d)} review')

Ngày chạy: 15/11/2026 | commit: 2927f88af499d702a5bd8145977e8f06411aca57
Train: 1200 review
Validation: 350 review
Hold-out: 1000 review
Focused: 150 review
Lazada: 500 review
Train ∪ Validation: 1550 review


## Kiểm tra trùng lặp nội dung

**Cách so khớp**

Văn bản được chuẩn hóa (NFC, chữ thường, bỏ dấu câu, gộp khoảng trắng) rồi so khớp chính xác. Bảng cho biết số văn bản
chung, số dòng của tập kiểm tra có văn bản xuất hiện ở tập tham chiếu, số dòng trùng mang nhãn actionable, độ dài tối đa
tính theo chữ và tỷ lệ dòng trùng có cùng nhãn với tập tham chiếu.

**Near-duplicate**

Cosine ≥ 0,90 trên TF-IDF ký tự 3–5-gram mà không trùng chính xác. Con số này chỉ được báo cáo, không dùng để loại review.

In [2]:
PAIRS = [('Train', 'Validation'), ('Train', 'Hold-out'), ('Validation', 'Hold-out'), ('Train ∪ Validation', 'Hold-out'),
         ('Train ∪ Validation', 'Focused'), ('Train', 'Lazada'), ('Train ∪ Validation', 'Lazada'), ('Hold-out', 'Lazada')]
DUP = PA.duplicate_audit(D, PAIRS)
show = DUP.copy()
show['tỷ lệ dòng trùng'] = (show['tỷ lệ dòng trùng'] * 100).round(1)
show['nhãn khớp tập tham chiếu'] = (show['nhãn khớp tập tham chiếu'] * 100).round(1)
print('Trùng chính xác giữa các tập (tỷ lệ theo %):'); print(show.to_string(index=False))

WITHIN = PA.within_duplicates({k: D[k] for k in SETS})
print('\nTrùng nội bộ từng tập:'); print(WITHIN.to_string(index=False))

NEAR = []
for ref, tgt in [('Train ∪ Validation', 'Hold-out'), ('Train ∪ Validation', 'Focused'), ('Train ∪ Validation', 'Lazada')]:
    nd = PA.near_duplicates(D[ref]['text'], D[tgt]['text'], 0.90)
    act = D[tgt]['gold_label'].to_numpy() != 'OTHER'
    NEAR.append({'tập tham chiếu': ref, 'tập kiểm tra': tgt, 'trùng chính xác': int(nd['exact'].sum()),
                 'near-duplicate (cosine ≥ 0,90)': int(nd['near'].sum()), 'near-duplicate actionable': int((nd['near'] & act).sum()),
                 'số chữ tối đa của near-duplicate': int(D[tgt]['text'].map(PA.normalize_text)[nd['near'].to_numpy()].str.split().str.len().max() or 0)
                 if nd['near'].any() else 0})
NEAR = pd.DataFrame(NEAR)
print('\nNear-duplicate:'); print(NEAR.to_string(index=False))

TOP = PA.top_shared(D['Train ∪ Validation'], D['Hold-out'], 10)
print('\nVăn bản trùng nhiều nhất giữa Train ∪ Validation và Hold-out:'); print(TOP.to_string(index=False))

Trùng chính xác giữa các tập (tỷ lệ theo %):
    tập tham chiếu tập kiểm tra  n tập kiểm tra  văn bản chung  dòng trùng  tỷ lệ dòng trùng  dòng trùng actionable  số chữ tối đa  nhãn khớp tập tham chiếu
             Train   Validation             350             21          83              23.7                      0              4                     100.0
             Train     Hold-out            1000             40         213              21.3                      1              4                     100.0
        Validation     Hold-out            1000             22         180              18.0                      1              4                     100.0
Train ∪ Validation     Hold-out            1000             44         217              21.7                      2              4                     100.0
Train ∪ Validation      Focused             150              1           1               0.7                      0              2                       0.0
             


Near-duplicate:
    tập tham chiếu tập kiểm tra  trùng chính xác  near-duplicate (cosine ≥ 0,90)  near-duplicate actionable  số chữ tối đa của near-duplicate
Train ∪ Validation     Hold-out              217                              17                          3                                 4
Train ∪ Validation      Focused                1                               2                          1                                 3
Train ∪ Validation       Lazada               79                               3                          1                                 3

Văn bản trùng nhiều nhất giữa Train ∪ Validation và Hold-out:
văn bản (chuẩn hóa)  số dòng ở tập kiểm tra  số dòng ở tập tham chiếu
                tốt                      57                        68
                 ok                      47                        87
               good                      13                        11
          tuyệt vời                      12                        17
  

## Tái tạo số GĐ10 trên tập đầy đủ

**Cách dựng dự đoán**

Giống notebook 12: tầng 1 neural lấy P(ACTIONABLE) so với ngưỡng frozen theo từng seed, tầng 2 và baseline phẳng lấy
argmax, LLM lấy nhãn đã lưu. Cascade ghép tầng 1 và tầng 2 cùng chiến lược, hybrid ghép tầng 1 của X với tầng 2 của Y
ở lần chạy chuẩn (seed 42 / LLM).

**Đối chiếu**

Bảng tầng 1, tầng 2, end-to-end và hybrid của Hold-out cùng bảng end-to-end của Lazada phải trùng từng ô với file `gd10_*`.

In [3]:
LBL = {l: i for i, l in enumerate(FOUR_CLASSES)}
NAME = {'holdout': 'Hold-out', 'lazada': 'Lazada'}
IDS = {s: D[n]['review_id'].tolist() for s, n in NAME.items()}
Y4 = {s: D[n]['gold_label'].map(LBL).to_numpy() for s, n in NAME.items()}
P = {s: PA.saved_predictions(RES, s, IDS[s], FZ) for s in NAME}
CFG, SKIP = {}, {}
for s in NAME:
    CFG[s], SKIP[s] = PA.configurations(*P[s])
    print(f'{NAME[s]}: {len(CFG[s])} cấu hình | hybrid bỏ qua vì thiếu dự đoán tầng 2: {SKIP[s] or "không"}')

def check(new, ref_file, key, cols, ref=None):
    ref = (pd.read_csv(os.path.join(RES, ref_file)) if ref is None else ref).set_index(key)[cols].astype(str)
    got = new.set_index(key).loc[ref.index, cols].astype(str)
    bad = (got != ref)
    assert not bad.any().any(), f'{ref_file}: lệch\n{pd.concat([got[bad.any(axis=1)], ref[bad.any(axis=1)]], axis=1)}'
    print(f'{ref_file}: khớp {len(ref)} dòng × {len(cols)} cột')

T1H = PA.tier1_scores(Y4['holdout'], P['holdout'][0])
check(T1H[0], 'gd10_tier1_holdout.csv', 'chiến lược', ['F1 ACT', 'Recall ACT', 'Precision ACT', 'Macro-F1 (2 lớp)'])
T2H = PA.tier2_scores(Y4['holdout'], P['holdout'][1])
check(T2H[0], 'gd10_tier2_holdout.csv', 'chiến lược', ['Macro-F1', 'F1 BUG', 'F1 FEATURE', 'F1 UX'])
E2E_H = PA.e2e_scores(Y4['holdout'], CFG['holdout'])
check(E2E_H[0], 'gd10_e2e_holdout.csv', 'cấu hình', ['Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER'])
HYB = pd.read_csv(os.path.join(RES, 'gd10_hybrid_holdout.csv'))
HYB['cấu hình'] = [PA.hybrid_name(x, y) for x, y in zip(HYB['tầng 1 (X)'], HYB['tầng 2 (Y)'])]
hy = E2E_H[0].assign(**{'Macro-F1 (4 lớp)': (E2E_H[0]['Macro-F1 chuẩn'] * 100).round(1),
                        'CI 95%': [f'{a * 100:.1f}–{b * 100:.1f}' for a, b in zip(E2E_H[0]['CI 2,5%'], E2E_H[0]['CI 97,5%'])]})
check(hy, 'gd10_hybrid_holdout.csv', 'cấu hình', ['Macro-F1 (4 lớp)', 'CI 95%'], ref=HYB)
E2E_L = PA.e2e_scores(Y4['lazada'], CFG['lazada'])
check(E2E_L[0], 'gd10_e2e_lazada.csv', 'cấu hình', ['Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER'])

Hold-out: 10 cấu hình | hybrid bỏ qua vì thiếu dự đoán tầng 2: không
Lazada: 10 cấu hình | hybrid bỏ qua vì thiếu dự đoán tầng 2: không


gd10_tier1_holdout.csv: khớp 3 dòng × 4 cột
gd10_tier2_holdout.csv: khớp 3 dòng × 4 cột


gd10_e2e_holdout.csv: khớp 4 dòng × 5 cột
gd10_hybrid_holdout.csv: khớp 9 dòng × 2 cột


gd10_e2e_lazada.csv: khớp 3 dòng × 5 cột


## Chấm lại trên tập khử trùng

**Tập khử trùng**

Bỏ các review Hold-out và Lazada có văn bản chuẩn hóa trùng với Train ∪ Validation, tức dữ liệu mà mô hình và prompt đã thấy.
Dự đoán giữ nguyên, chỉ tập được chấm thu hẹp lại.

**Cách đọc**

Chênh lệch là giá trị trên tập khử trùng trừ giá trị trên tập đầy đủ, theo điểm phần trăm. Mô hình neural lấy trung bình
qua ba seed; khoảng tin cậy 95% tính trên lần chạy chuẩn bằng bootstrap 10.000 lần.

In [4]:
KEEP = {s: PA.keep_mask(D[n]['text'], D['Train ∪ Validation']['text']) for s, n in NAME.items()}
for s, n in NAME.items():
    act = Y4[s] != LBL['OTHER']
    print(f'{n}: giữ {KEEP[s].sum()}/{len(KEEP[s])} review | bỏ {(~KEEP[s]).sum()} (actionable bị bỏ: {(~KEEP[s] & act).sum()}) '
          f'| actionable còn lại: {(KEEP[s] & act).sum()}')

T1HD = PA.tier1_scores(Y4['holdout'], P['holdout'][0], KEEP['holdout'])
T2HD = PA.tier2_scores(Y4['holdout'], P['holdout'][1], KEEP['holdout'])
E2E_HD = PA.e2e_scores(Y4['holdout'], CFG['holdout'], KEEP['holdout'])

def compare(full, dd, key, col, tier):
    out = full[[key, col]].merge(dd[[key, col]], on=key, suffixes=(' đầy đủ', ' khử trùng'))
    out = out.rename(columns={key: 'cấu hình'})
    out.insert(0, 'chỉ số', f'{tier}: {col.replace(" (trung bình)", "")}')
    out = out.rename(columns={f'{col} đầy đủ': 'đầy đủ', f'{col} khử trùng': 'khử trùng'})
    out['chênh (điểm %)'] = ((out['khử trùng'] - out['đầy đủ']) * 100).round(2)
    out[['đầy đủ', 'khử trùng']] = (out[['đầy đủ', 'khử trùng']] * 100).round(1)
    return out

CMP_H = pd.concat([compare(T1H[0], T1HD[0], 'chiến lược', 'F1 ACT (trung bình)', 'Tầng 1'),
                   compare(T1H[0], T1HD[0], 'chiến lược', 'Recall ACT (trung bình)', 'Tầng 1'),
                   compare(T2H[0], T2HD[0], 'chiến lược', 'Macro-F1 (trung bình)', 'Tầng 2'),
                   compare(E2E_H[0], E2E_HD[0], 'cấu hình', 'Macro-F1 (trung bình)', 'End-to-end')], ignore_index=True)
print('Hold-out — đầy đủ và khử trùng (%):'); print(CMP_H.to_string(index=False))
print('\nEnd-to-end trên Hold-out khử trùng:')
print(E2E_HD[0][['cấu hình', 'n', 'Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER', 'số lần chạy']].to_string(index=False))

Hold-out: giữ 783/1000 review | bỏ 217 (actionable bị bỏ: 2) | actionable còn lại: 181
Lazada: giữ 421/500 review | bỏ 79 (actionable bị bỏ: 1) | actionable còn lại: 80


Hold-out — đầy đủ và khử trùng (%):
              chỉ số                          cấu hình  đầy đủ  khử trùng  chênh (điểm %)
      Tầng 1: F1 ACT                          ViSoBERT    65.9       65.6           -0.36
      Tầng 1: F1 ACT                Bi-LSTM + fastText    59.7       59.7           -0.03
      Tầng 1: F1 ACT                      LLM few-shot    81.7       81.8            0.02
  Tầng 1: Recall ACT                          ViSoBERT    69.8       69.4           -0.33
  Tầng 1: Recall ACT                Bi-LSTM + fastText    84.0       83.8           -0.18
  Tầng 1: Recall ACT                      LLM few-shot    86.9       86.7           -0.14
    Tầng 2: Macro-F1                          ViSoBERT    76.5       76.3           -0.14
    Tầng 2: Macro-F1                Bi-LSTM + fastText    69.6       69.4           -0.21
    Tầng 2: Macro-F1                      LLM few-shot    89.7       89.6           -0.08
End-to-end: Macro-F1                  Cascade ViSoBERT    62.1  

## Baseline phẳng và hybrid trên Lazada

**Nội dung**

Dự đoán của baseline phẳng ViSoBERT trên Lazada đã được lưu ở phiên 10a nhưng GĐ10 chưa chấm. Phần này chấm cùng ba
cascade và các hybrid có đủ dự đoán tầng 2, trên tập đầy đủ và tập khử trùng, rồi so mức suy giảm từ Hold-out sang Lazada.

In [5]:
E2E_LD = PA.e2e_scores(Y4['lazada'], CFG['lazada'], KEEP['lazada'])
T1L = PA.tier1_scores(Y4['lazada'], P['lazada'][0])
cols = ['cấu hình', 'n', 'Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER', 'số lần chạy']
print('End-to-end — Lazada đầy đủ (%):'); print(E2E_L[0][cols].to_string(index=False))
print('\nEnd-to-end — Lazada khử trùng (%):'); print(E2E_LD[0][cols].to_string(index=False))
print('\nTầng 1 — Lazada (%):'); print(T1L[0][['chiến lược', 'F1 ACT', 'Recall ACT', 'Precision ACT', 'Macro-F1 (2 lớp)']].to_string(index=False))
d = E2E_L[1]
print('\nChênh Macro-F1 có cặp liên quan baseline phẳng, Lazada (seed 42 / LLM, bootstrap ghép cặp):')
print(d[(d['A'] == PA.FLAT) | (d['B'] == PA.FLAT)].round(4).to_string(index=False))

DROP = E2E_H[0][['cấu hình', 'Macro-F1 (trung bình)']].merge(E2E_L[0][['cấu hình', 'Macro-F1 (trung bình)']], on='cấu hình',
                                                              suffixes=(' Hold-out', ' Lazada'))
DROP['chênh (điểm %)'] = ((DROP['Macro-F1 (trung bình) Lazada'] - DROP['Macro-F1 (trung bình) Hold-out']) * 100).round(1)
DROP[['Macro-F1 (trung bình) Hold-out', 'Macro-F1 (trung bình) Lazada']] = (DROP[['Macro-F1 (trung bình) Hold-out', 'Macro-F1 (trung bình) Lazada']] * 100).round(1)
print('\nSuy giảm từ Hold-out sang Lazada (end-to-end, %):'); print(DROP.to_string(index=False))

End-to-end — Lazada đầy đủ (%):
                         cấu hình   n Macro-F1 (4 lớp)     F1 BUG F1 FEATURE      F1 UX   F1 OTHER  số lần chạy
                 Cascade ViSoBERT 500       56.0 ± 0.4 42.4 ± 1.6 33.8 ± 2.1 59.0 ± 2.4 88.8 ± 1.3            3
       Cascade Bi-LSTM + fastText 500       47.1 ± 6.4 31.9 ± 5.5 21.3 ± 6.7 55.3 ± 5.9 80.0 ± 8.3            3
             Cascade LLM few-shot 500             60.5       48.1       37.5       66.7       89.8            1
          Baseline phẳng ViSoBERT 500       65.3 ± 2.5 50.3 ± 2.0 43.2 ± 2.8 74.5 ± 6.1 93.4 ± 0.6            3
    ViSoBERT → Bi-LSTM + fastText 500             52.8       35.3       25.5       61.5       89.0            1
          ViSoBERT → LLM few-shot 500             57.9       44.4       40.0       58.1       89.0            1
    Bi-LSTM + fastText → ViSoBERT 500             50.7       39.2       28.6       50.8       84.1            1
Bi-LSTM + fastText → LLM few-shot 500             54.9       43.1       

## Kiểm định McNemar

**Cách kiểm**

Với mỗi cặp cấu hình, đếm số review cấu hình A đúng còn B sai và ngược lại, rồi kiểm định nhị thức chính xác hai phía.
p được hiệu chỉnh Holm trong từng họ so sánh. McNemar kiểm tỷ lệ đúng, bổ sung cho bootstrap chênh Macro-F1 vốn nhạy hơn
với các lớp hiếm.

In [6]:
MC = pd.concat([PA.mcnemar_table(T1H[1], T1H[2], 'tầng 1 – Hold-out'),
                PA.mcnemar_table(T2H[1], T2H[2], 'tầng 2 – Hold-out (gold-actionable)'),
                PA.mcnemar_table(E2E_H[2], E2E_H[3], 'end-to-end – Hold-out'),
                PA.mcnemar_table(E2E_L[2], E2E_L[3], 'end-to-end – Lazada')], ignore_index=True)
print(MC.groupby('họ so sánh')['khác biệt (α = 0,05)'].agg(['size', 'sum']).rename(columns={'size': 'số cặp', 'sum': 'có ý nghĩa'}))
main = ['Cascade ViSoBERT', 'Cascade Bi-LSTM + fastText', 'Cascade LLM few-shot', PA.FLAT, 'ViSoBERT → LLM few-shot']
sel = MC[MC['họ so sánh'].str.startswith('tầng') | (MC['A'].isin(main) & MC['B'].isin(main))].copy()
sel[['tỷ lệ đúng A', 'tỷ lệ đúng B']] = (sel[['tỷ lệ đúng A', 'tỷ lệ đúng B']] * 100).round(1)
print(); print(sel.round({'p (exact)': 4, 'p (Holm)': 4}).to_string(index=False))

                                     số cặp  có ý nghĩa
họ so sánh                                             
end-to-end – Hold-out                    45          35
end-to-end – Lazada                      45          22
tầng 1 – Hold-out                         3           3
tầng 2 – Hold-out (gold-actionable)       3           2

                         họ so sánh                          A                          B  tỷ lệ đúng A  tỷ lệ đúng B  A đúng, B sai  A sai, B đúng  p (exact)  p (Holm)  khác biệt (α = 0,05)
                  tầng 1 – Hold-out                   ViSoBERT         Bi-LSTM + fastText          86.6          81.5            107             56     0.0001    0.0001                  True
                  tầng 1 – Hold-out                   ViSoBERT               LLM few-shot          86.6          92.9             41            104     0.0000    0.0000                  True
                  tầng 1 – Hold-out         Bi-LSTM + fastText               LLM few-shot 

## Phân tích lỗi theo lớp

**Phân rã lỗi end-to-end**

Mỗi lỗi bốn lớp thuộc một trong ba dạng: bỏ sót actionable (gold BUG/FEATURE/UX bị gán OTHER), giữ nhầm OTHER (gold OTHER
bị gán BUG/FEATURE/UX) và sai loại (cả gold và dự đoán đều actionable nhưng khác loại). Số lỗi là trung bình qua các lần chạy.

**Theo lớp**

Precision, recall và F1 từng lớp của mọi cấu hình, kèm ma trận nhầm lẫn của lần chạy chuẩn.

In [7]:
ERR_H = PA.error_decomposition(Y4['holdout'], CFG['holdout'])
ERR_L = PA.error_decomposition(Y4['lazada'], CFG['lazada'])
print('Phân rã lỗi — Hold-out:'); print(ERR_H.to_string(index=False))
print('\nPhân rã lỗi — Lazada:'); print(ERR_L.to_string(index=False))
PC_H = PA.per_class(Y4['holdout'], CFG['holdout'])
print('\nRecall và F1 từng lớp — Hold-out (%):')
print(PC_H.pivot(index='cấu hình', columns='lớp', values='recall')[FOUR_CLASSES].add_prefix('recall ').join(
      PC_H.pivot(index='cấu hình', columns='lớp', values='F1')[FOUR_CLASSES].add_prefix('F1 ')).loc[list(CFG['holdout'])].to_string())
CONF_H = PA.confusion_long(Y4['holdout'], CFG['holdout'])
off = CONF_H.melt(id_vars=['cấu hình', 'gold'], var_name='đoán', value_name='số review')
off = off[off['gold'].str.replace('gold ', '') != off['đoán'].str.replace('đoán ', '')]
print('\nBa cặp nhầm lẫn lớn nhất của mỗi cấu hình (lần chạy chuẩn):')
print(off.sort_values('số review', ascending=False).groupby('cấu hình').head(3).sort_values(['cấu hình', 'số review'], ascending=[True, False]).to_string(index=False))

Phân rã lỗi — Hold-out:
                         cấu hình    n  tổng lỗi  bỏ sót actionable  giữ nhầm OTHER  sai loại  % bỏ sót actionable  % giữ nhầm OTHER  % sai loại
                 Cascade ViSoBERT 1000     151.0               55.3            76.7      19.0                 36.6              50.8        12.6
       Cascade Bi-LSTM + fastText 1000     250.0               29.3           182.7      38.0                 11.7              73.1        15.2
             Cascade LLM few-shot 1000      80.0               24.0            47.0       9.0                 30.0              58.8        11.2
          Baseline phẳng ViSoBERT 1000     108.7               74.0            20.7      14.0                 68.1              19.0        12.9
    ViSoBERT → Bi-LSTM + fastText 1000     160.0               59.0            75.0      26.0                 36.9              46.9        16.2
          ViSoBERT → LLM few-shot 1000     144.0               59.0            75.0      10.0             

lớp                                recall BUG recall FEATURE   recall UX recall OTHER      F1 BUG  F1 FEATURE       F1 UX    F1 OTHER
cấu hình                                                                                                                             
Cascade ViSoBERT                   46.8 ± 6.4     45.8 ± 7.2  80.0 ± 2.9   90.6 ± 1.7  48.5 ± 2.0  34.8 ± 3.6  73.2 ± 0.4  91.8 ± 0.5
Cascade Bi-LSTM + fastText         58.1 ± 9.7     43.1 ± 8.7  76.7 ± 4.1   77.6 ± 9.2  40.2 ± 2.1  22.2 ± 7.3  66.1 ± 3.6  85.5 ± 5.2
Cascade LLM few-shot                     82.0           79.2        82.9         94.2        71.9        69.1        88.5        95.6
Baseline phẳng ViSoBERT            36.3 ± 0.6     40.3 ± 2.4  75.7 ± 1.4   97.5 ± 0.6  47.0 ± 1.4  46.1 ± 3.6  79.7 ± 0.8  94.4 ± 0.3
ViSoBERT → Bi-LSTM + fastText            38.2           41.7        77.1         90.8        40.5        29.9        73.5        91.7
ViSoBERT → LLM few-shot                  47.2           70.8  

## Biên Pareto hiệu năng – chi phí

**Dữ liệu**

Macro-F1 end-to-end của lần chạy chuẩn trên Hold-out đầy đủ, kèm khoảng tin cậy 95%. Chi phí lấy từ GĐ10: thời gian suy
luận và chi phí API cho 1.000 review, với cascade và hybrid gồm tầng 1 trên mọi review cộng tầng 2 trên phần được giữ lại.

**Biên Pareto**

Một cấu hình nằm trên biên khi không cấu hình nào khác vừa có Macro-F1 không thấp hơn vừa có chi phí không cao hơn. Biên
được tính riêng cho từng thước đo chi phí. Hình chỉ vẽ khoảng tin cậy của các điểm trên biên, khoảng tin cậy của mọi cấu
hình nằm trong `paper_pareto.csv`.

In [8]:
CS = pd.read_csv(os.path.join(RES, 'gd10_cost_summary.csv')).set_index('chiến lược')
COST = {n: (s, u) for n, s, u in zip(HYB['cấu hình'], HYB['giây / 1.000 review'], HYB['USD API / 1.000 review'])}
COST[PA.FLAT] = (CS.loc['Baseline phẳng ViSoBERT', 'suy luận / 1.000 review (giây)'], CS.loc['Baseline phẳng ViSoBERT', 'USD API / 1.000 review'])
PAR = E2E_H[0][['cấu hình', 'Macro-F1 chuẩn', 'CI 2,5%', 'CI 97,5%']].rename(columns={'Macro-F1 chuẩn': 'Macro-F1'})
PAR['giây / 1.000 review'] = PAR['cấu hình'].map(lambda n: COST[n][0])
PAR['USD API / 1.000 review'] = PAR['cấu hình'].map(lambda n: COST[n][1])
PAR['Pareto (USD)'] = PA.pareto_front(PAR['Macro-F1'], PAR['USD API / 1.000 review'])
PAR['Pareto (giây)'] = PA.pareto_front(PAR['Macro-F1'], PAR['giây / 1.000 review'])
PAR.insert(1, 'mã', PAR['cấu hình'].map(PA.short_label))
show = PAR.sort_values('USD API / 1.000 review').copy()
show[['Macro-F1', 'CI 2,5%', 'CI 97,5%']] = (show[['Macro-F1', 'CI 2,5%', 'CI 97,5%']] * 100).round(1)
print(show.to_string(index=False))

TXT = {'vi': {'y': 'Macro-F1 end-to-end 4 lớp (%)', 'cascade': 'Cascade thuần', 'phẳng': 'Phẳng 4 lớp',
              'hybrid': 'Hybrid (tầng 1 → tầng 2)', 'front': 'Biên Pareto', 'dominated': 'Bị trội', 'flat_code': 'V phẳng',
              'x': ['Chi phí API (USD / 1.000 review)', 'Thời gian suy luận (giây / 1.000 review, thang log)']},
       'en': {'y': 'End-to-end Macro-F1, 4 classes (%)', 'cascade': 'Pure cascade', 'phẳng': 'Flat 4-class',
              'hybrid': 'Hybrid (tier 1 → tier 2)', 'front': 'Pareto front', 'dominated': 'Dominated', 'flat_code': 'V flat',
              'x': ['API cost (USD per 1,000 reviews)', 'Inference time (s per 1,000 reviews, log scale)']}}
FIGS = []
for lang, t in TXT.items():
    panels = [('USD API / 1.000 review', t['x'][0], False, 'Pareto (USD)'), ('giây / 1.000 review', t['x'][1], True, 'Pareto (giây)')]
    FIGS.append(PA.pareto_figure(PAR, os.path.join(RES, f'paper_pareto_{lang}.png'), panels, t))
print('Hình:', [os.path.basename(f) for f in FIGS])

                         cấu hình      mã  Macro-F1  CI 2,5%  CI 97,5%  giây / 1.000 review  USD API / 1.000 review  Pareto (USD)  Pareto (giây)
                 Cascade ViSoBERT     V→V      62.9     57.6      67.8                 1.55                  0.0000         False          False
       Cascade Bi-LSTM + fastText     B→B      54.2     49.9      58.3                 0.11                  0.0000         False           True
          Baseline phẳng ViSoBERT V phẳng      66.7     60.3      72.1                 1.39                  0.0000          True           True
    ViSoBERT → Bi-LSTM + fastText     V→B      58.9     53.6      63.7                 1.28                  0.0000         False          False
    Bi-LSTM + fastText → ViSoBERT     B→V      58.9     54.2      63.4                 0.51                  0.0000         False           True
          ViSoBERT → LLM few-shot     V→L      69.5     63.9      74.4               185.17                  0.0320          True 

Hình: ['paper_pareto_vi.png', 'paper_pareto_en.png']


## Lưu kết quả

**Các file**

Mọi bảng được lưu với tiền tố `paper_` trong `05_experiments/rq1_classification/`, cùng hai hình Pareto (tiếng Việt cho
khóa luận, tiếng Anh cho bài báo) và `paper_run_info.json` ghi ngày chạy và commit.

In [9]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'paper_{name}.csv'), index=False, encoding='utf-8-sig')

save(DUP, 'duplicates_between'); save(WITHIN, 'duplicates_within'); save(NEAR, 'near_duplicates'); save(TOP, 'duplicates_top_holdout')
save(CMP_H, 'dedup_compare_holdout'); save(T1HD[0], 'dedup_tier1_holdout'); save(T2HD[0], 'dedup_tier2_holdout'); save(E2E_HD[0], 'dedup_e2e_holdout')
save(E2E_H[0], 'e2e_holdout_all'); save(E2E_H[1], 'e2e_holdout_diff')
save(E2E_L[0], 'e2e_lazada_all'); save(E2E_L[1], 'e2e_lazada_diff'); save(E2E_LD[0], 'dedup_e2e_lazada'); save(T1L[0], 'tier1_lazada'); save(DROP, 'holdout_vs_lazada')
save(MC, 'mcnemar')
save(PC_H, 'per_class_holdout'); save(ERR_H, 'errors_holdout'); save(ERR_L, 'errors_lazada'); save(CONF_H, 'confusion_holdout')
save(PAR, 'pareto')
json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'python': sys.version.split()[0], 'amendment': 'v3.1', 'n_boot': 10_000, 'bootstrap_seed': 42,
           'skipped_hybrids': {s: v for s, v in SKIP.items()}}, open(os.path.join(RES, 'paper_run_info.json'), 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith('paper_')))

Đã lưu: ['paper_confusion_holdout.csv', 'paper_dedup_compare_holdout.csv', 'paper_dedup_e2e_holdout.csv', 'paper_dedup_e2e_lazada.csv', 'paper_dedup_tier1_holdout.csv', 'paper_dedup_tier2_holdout.csv', 'paper_duplicates_between.csv', 'paper_duplicates_top_holdout.csv', 'paper_duplicates_within.csv', 'paper_e2e_holdout_all.csv', 'paper_e2e_holdout_diff.csv', 'paper_e2e_lazada_all.csv', 'paper_e2e_lazada_diff.csv', 'paper_errors_holdout.csv', 'paper_errors_lazada.csv', 'paper_holdout_vs_lazada.csv', 'paper_mcnemar.csv', 'paper_near_duplicates.csv', 'paper_pareto.csv', 'paper_pareto_en.png', 'paper_pareto_vi.png', 'paper_per_class_holdout.csv', 'paper_run_info.json', 'paper_tier1_lazada.csv']


## Nhận xét

**Nguồn số liệu**

Mọi số dưới đây lấy từ các file `paper_*` của lần chạy này. Mô hình neural là trung bình ± độ lệch chuẩn qua ba seed,
khoảng tin cậy và kiểm định tính trên lần chạy chuẩn (seed 42 / LLM).

**Trùng lặp nội dung**

217/1.000 review Hold-out (21,7%) có văn bản chuẩn hóa trùng Train ∪ Validation, tất cả dài 1–4 chữ như "tốt", "ok",
"good", chỉ 2 review là actionable và nhãn khớp hoàn toàn với tập tham chiếu. Trùng theo review_id bằng 0, near-duplicate
cosine ≥ 0,90 thêm 17 review. Đây là trùng tự nhiên của các review khen ngắn, không phải rò rỉ do quy trình chia tập.

**Tác động của khử trùng**

Trên 783 review Hold-out còn lại, chỉ số tầng 1 và tầng 2 đổi không quá 0,4 điểm, Macro-F1 end-to-end giảm 0,4–1,6 điểm
vì các review OTHER dễ bị bỏ đi. Trên 421 review Lazada còn lại, Macro-F1 giảm 0,6–1,8 điểm. Thứ hạng chỉ đổi giữa các cặp
vốn chênh nhau không quá 0,3 điểm.

**Baseline phẳng trên Lazada**

Baseline phẳng ViSoBERT đạt 65,3 ± 2,5 trên Lazada, cao nhất trong mười cấu hình, và chỉ giảm 1,5 điểm so với Hold-out.
Cascade LLM giảm 20,8 điểm (81,3 xuống 60,5), cascade ViSoBERT giảm 6,1 điểm. Chênh giữa baseline phẳng và cascade LLM
trên Lazada là +5,1 điểm với khoảng tin cậy từ −4,5 đến +14,9, chưa đủ để khẳng định khác biệt.

**Nguồn lỗi**

Sai loại trong nhóm actionable chỉ chiếm 5–16% lỗi end-to-end, trừ hai hybrid có tầng 1 LLM và tầng 2 neural (26–31%).
Phần lớn lỗi nằm ở ranh giới actionable/OTHER, với nhầm lẫn lớn nhất là OTHER thành BUG và BUG thành OTHER. Cascade chủ yếu
giữ nhầm OTHER do ngưỡng tầng 1 ưu tiên recall, còn baseline phẳng chủ yếu bỏ sót actionable: recall actionable suy ra từ
số bỏ sót là 59,6% ở baseline phẳng, so với 69,8% ở cascade ViSoBERT và 86,9% ở cascade LLM.

**Kiểm định**

35/45 cặp end-to-end trên Hold-out khác biệt có ý nghĩa theo McNemar sau hiệu chỉnh Holm. Cascade ViSoBERT kém baseline phẳng
về tỷ lệ đúng (84,9% và 88,9%, p Holm < 0,001) trong khi chênh Macro-F1 −3,8 điểm có khoảng tin cậy chứa 0. Ngược lại,
cascade LLM hơn baseline phẳng 14,6 điểm Macro-F1 (khoảng tin cậy +8,6 đến +21,0) nhưng chênh tỷ lệ đúng không có ý nghĩa
(p Holm 0,070), vì tỷ lệ đúng bị lớp OTHER chiếm 81,7% Hold-out chi phối. Tầng 2 giữa ViSoBERT và Bi-LSTM không khác biệt
có ý nghĩa (p 0,099).

**Biên Pareto**

Theo chi phí API, biên gồm baseline phẳng ViSoBERT (66,7; 0 USD), ViSoBERT → LLM (69,5; 0,032 USD cho 1.000 review),
LLM → ViSoBERT (73,3; 0,173 USD) và cascade LLM (81,3; 0,206 USD). Hybrid ViSoBERT → LLM chỉ hơn baseline phẳng 2,8 điểm
với khoảng tin cậy chứa 0, nên bước cải thiện chắc chắn duy nhất trên Hold-out là chuyển sang cascade LLM.

**Hệ quả cho các bước sau**

Trùng tự nhiên không giải thích được kết quả. Kiến trúc phẳng và cascade khác nhau chủ yếu ở điểm vận hành giữa bỏ sót và
giữ nhầm actionable, nên so sánh kiến trúc cần báo cả Macro-F1 lẫn recall actionable. Mức giảm lớn của LLM trên Lazada cần
thí nghiệm prompt trung tính để tách phần do prompt viết riêng cho Shopee.